# E-commerce Sales Analytics — PySpark ETL Project

This is my end-to-end ETL + analytics project on Databricks. I start with raw e-commerce CSVs, fix a real data-quality issue, join all four tables with SQL, run four phases of business analysis, and finish with a Bronze/Silver/Gold Delta Lake setup in Unity Catalog.

## Step 1: Reading the raw data

I load all four raw CSVs from a Unity Catalog volume and take a first look at each one.

In [0]:
df1=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/workspace/default/tutorial_learning/raw/customers.csv")

In [0]:
df1.display()

In [0]:
df2=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/workspace/default/tutorial_learning/raw/orders.csv")
df2.display()

In [0]:
df3=spark.read.format("csv").option("header",True).option("inferSchema",True).load("/Volumes/workspace/default/tutorial_learning/raw/order_items.csv")
df3.display()

In [0]:
df4=spark.read.format("csv").option("header",True).option("inferSchema",True).load("/Volumes/workspace/default/tutorial_learning/raw/products.csv")
df4.display()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### Step 2: Order value rollup — one total price per order

Order items by themselves don't tell me what an order is actually worth, so I roll them up to the order level first.

In [0]:
df_ord_price=df3.groupBy('order_id').agg(round(sum(col('unit_price')*col('quantity')),0).alias("total_price"))
df_ord_price.display()

In [0]:
df_ord_price.printSchema()

### Step 3: Validating data types (schema checks)

I print each schema to make sure dates came in as dates, prices as doubles, and IDs as integers — before I trust any of the numbers.

In [0]:
df1.printSchema()

In [0]:
df2.printSchema()

In [0]:
df3.printSchema()

In [0]:
df4.printSchema()

### Step 4: Checking duplicates across all four tables

Now I compare row counts vs distinct counts to see if there are any duplicates. This is actually where I found a real data-quality issue.

In [0]:
df1.createOrReplaceTempView("customers")
df2.createOrReplaceTempView("orders")
df3.createOrReplaceTempView("order_items")
df4.createOrReplaceTempView("products")
df_ord_price.createOrReplaceTempView("order_price")

In [0]:
print("customers",df1.count())
print("orders",df2.count())
print("order_items",df3.count())
print("products",df4.count())

In [0]:
print("customers",df1.distinct().count())
print("orders",df2.distinct().count())
print("order_items",df3.distinct().count())
print("products",df4.distinct().count())

#### Found: 15 duplicate customer records

So the row count is 2,015 but there are only 2,000 distinct customer_ids — that means 15 duplicates. Let me find which customer_ids are repeating:

In [0]:
spark.sql("select customer_id, count(customer_id) as count from customers group by customer_id having count(customer_id) > 1").display()

#### Data-quality fix: I deduplicate the customers table

Those 15 duplicate customer_ids would mess up every join below — they'd inflate revenue by about $164K. So I'm keeping just one row per customer and refreshing the temp view so everything going forward uses the clean data.

In [0]:
# keep ONE row per customer_id
before = df1.count()
df1 = df1.dropDuplicates(["customer_id"])
after = df1.count()
print(f"customers: {before} rows -> {after} rows (removed {before - after} duplicates)")

# refresh the temp view so the SQL joins below use the DEDUPED table
df1.createOrReplaceTempView("customers")

### Step 5: Joining all tables with SQL temp views

Now I register each DataFrame as a temp view and join all four tables together — customers, orders, order_items, and products — into one big dataset.

In [0]:
df = spark.sql("""
select 
  c.customer_id,
  c.customer_name,
  c.city,
  c.state,
  c.country,
  c.signup_date,
  o.order_id,
  o.order_date,
  o.payment_method,
  o.order_status,
  i.product_id,
  i.quantity,
  i.unit_price,
  p.product_name,
  p.category,
  p.price
from customers as c
left join orders as o on c.customer_id = o.customer_id
left join order_items as i on o.order_id = i.order_id
left join products as p on i.product_id = p.product_id
""")
df.display()

### Phase 1: Order & sales analysis — revenue trends, AOV, status & payment mix

#### Order-level dataset (orders + order value)

I join orders with their rolled-up totals here and pull out year and month so I can do time-based analysis.

In [0]:
dfo = spark.sql("select o.*,op.total_price from orders as o left join order_price as op on o.order_id = op.order_id")
dfo.display()

In [0]:
dfo1=dfo.withColumn('month',month('order_date')).withColumn('year',year('order_date'))

In [0]:
dfo1.display()

#### Monthly revenue trend

Only paid orders count as revenue here: Shipped, Processing or Completed.

In [0]:
df_monthly=dfo1.filter(col('order_status').isin(['Shipped','Processing','Completed'])).groupby('year','month').agg(countDistinct('order_id').alias('total_orders'),sum('total_price').alias('total_revenue')).orderBy('year','month')

In [0]:
df_monthly.display()

#### Yearly revenue & average order value (AOV)

AOV = total revenue / total orders — what a typical order is worth each year.

In [0]:
df_yearly=dfo1.filter(col('year')<=2025).filter(col('order_status').isin(['Shipped','Processing','Completed'])).groupby('year').agg(sum('total_price').alias('total_revenue'),countDistinct('order_id').alias('total_orders')).orderBy('year')

In [0]:
df_yearly.withColumn('AOV',round(col('total_revenue')/col('total_orders'),2)).display()

#### Order status distribution

In [0]:
df.groupby('order_status').agg(count('customer_id').alias('count')).display()

#### Payment method analysis — which payment types drive revenue

In [0]:
dfo1.groupby('payment_method').agg(countDistinct('order_id').alias('total_orders'),sum('total_price').alias('total_revenue')).display()

### Phase 2: Product analysis — top products & category performance

In [0]:
dfo2 = spark.sql("select oi.*,o.order_status,p.product_name,p.category,p.price from order_items as oi left join orders as o on oi.order_id = o.order_id left join products as p on oi.product_id = p.product_id")
dfo2.display()

#### Top 10 products by revenue (paid orders only)

In [0]:
dfo2.filter(col('order_status').isin(['Shipped','Processing','Completed'])).groupby('product_name').agg(sum('quantity').alias('total_quantity'),round(sum(col('quantity')*col('unit_price')),0).alias('total_revenue')).orderBy(col('total_revenue').desc()).limit(10).display()

#### Category performance — ranked with a window function

In [0]:
from pyspark.sql.window import Window

In [0]:
dfo2.filter(col('order_status').isin(['Shipped','Processing','Completed'])).groupby('category').agg(sum('quantity').alias('total_quantity'),countDistinct('order_id').alias('total_orders'),round(sum(col('quantity')*col('unit_price')),0).alias('total_revenue')).withColumn('ranking',rank().over(Window.partitionBy().orderBy(col('total_revenue').desc()))).display()

### Phase 3: Customer analysis — top customers & regional sales

#### Overview of the joined data

In [0]:
df.display()

#### Top 10 customers by spending

In [0]:
df.filter(col('order_status').isin(['Shipped','Processing','Completed'])).groupby('customer_id','customer_name').agg(countDistinct('order_id').alias('total_orders'),round(sum((col('quantity')*col('unit_price'))),2).alias('total_spent')).orderBy(col('total_spent').desc()).limit(10).display()

#### Sales by state and city

In [0]:
df.filter(col('order_status').isin(['Shipped','Processing','Completed'])).groupby('state').agg(countDistinct('order_id').alias('total_orders'),sum('quantity').alias('total_quantity'),round(sum((col('quantity')*col('unit_price'))),0).alias('total_revenue')).orderBy(col('total_revenue').desc()).display()

In [0]:
df.filter(col('order_status').isin(['Shipped','Processing','Completed'])).groupby('city').agg(countDistinct('order_id').alias('total_orders'),sum('quantity').alias('total_quantity'),round(sum((col('quantity')*col('unit_price'))),0).alias('total_revenue')).orderBy(col('total_revenue').desc()).display()

### Phase 4: Advanced analysis — lost sales & customer cohorts

#### Cancelled orders & potential lost sales

In [0]:
df.filter(col('order_status').isin(['Cancelled','Completed'])).groupby('order_status').agg(countDistinct('order_id').alias('total_orders'),round(sum((col('quantity')*col('unit_price'))),0).alias('total_revenue')).display()

#### Customer signup cohorts & conversion rate

I check which signup months produced customers who actually ordered — and each cohort's conversion rate.

In [0]:
df_info = spark.sql("select c.*,order_id,o.order_date,o.payment_method,o.order_status from customers as c left join orders as o on c.customer_id = o.customer_id")
df_info.display()

In [0]:
df_info=df_info.withColumn('signup_month',month('signup_date')).withColumn('signupyear',year('signup_date'))

In [0]:
df_info.groupby('signupyear','signup_month').agg(countDistinct('customer_id').alias('total_customers')).orderBy('signupyear','signup_month').display()

In [0]:
df_info.groupby('signupyear', 'signup_month').agg(
    countDistinct('customer_id').alias('total_customers'),
    countDistinct(when(col('order_id').isNotNull(), col('customer_id'))).alias('customers_who_ordered')
).withColumn(
    'conversion_rate',
    round(col('customers_who_ordered') / col('total_customers') * 100, 2)
).orderBy('signupyear', 'signup_month').display()

### Phase 5: Medallion Architecture — Bronze → Silver → Gold

I'm saving every stage as real Delta tables in `workspace.default`, so the results stick around after the session and any dashboard or query can read them:

* 🥉 **Bronze** — raw data exactly as I got it (keeping duplicates for audit)
* 🥈 **Silver** — I removed duplicates and validated everything — this is the trusted layer
* 🥇 **Gold** — business aggregates like revenue, products, customers — ready for dashboards



In [0]:
# Bronze layer — this is the raw data as I got it, keeping duplicates for audit purposes
bronze_sources = {
    "bronze_customers":   "/Volumes/workspace/default/tutorial_learning/raw/customers.csv",
    "bronze_orders":      "/Volumes/workspace/default/tutorial_learning/raw/orders.csv",
    "bronze_order_items": "/Volumes/workspace/default/tutorial_learning/raw/order_items.csv",
    "bronze_products":    "/Volumes/workspace/default/tutorial_learning/raw/products.csv",
}

for table_name, path in bronze_sources.items():
    (spark.read.format("csv").option("header", True).option("inferSchema", True).load(path)
         .write.format("delta").mode("overwrite").saveAsTable(f"workspace.default.{table_name}"))
    print(f"{table_name:20s} written ({spark.table(f'workspace.default.{table_name}').count()} rows)")

In [0]:
# Silver layer — I cleaned this up by removing duplicates and validating the data 
(
    df1.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_customers"),
    df2.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_orders"),
    df3.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_order_items"),
    df4.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_products"),
)



In [0]:
# Gold layer — these are the final numbers I'd actually use for dashboards

VALID = col("order_status").isin(["Shipped", "Processing", "Completed"])


orders_value = (df2.join(df_ord_price, "order_id", "left")
                  .withColumn("year", year("order_date"))
                  .withColumn("month", month("order_date")))


line_items = (df3.join(df2.select("order_id", "customer_id", "order_status"), "order_id", "left")
                 .join(df4.select("product_id", "product_name", "category"), "product_id", "left")
                 .join(df1.select("customer_id", "customer_name", "state"), "customer_id", "left"))

# 1) monthly revenue
orders_value.filter(VALID).groupby("year", "month") \
    .agg(countDistinct("order_id").alias("total_orders"), sum("total_price").alias("total_revenue")) \
    .orderBy("year", "month") \
    .write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_monthly_revenue")

# 2) yearly revenue + average order value
orders_value.filter(VALID).groupby("year") \
    .agg(sum("total_price").alias("total_revenue"), countDistinct("order_id").alias("total_orders")) \
    .orderBy("year") \
    .withColumn("AOV", round(col("total_revenue") / col("total_orders"), 2)) \
    .write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_yearly_revenue")

# 3) top 10 products by revenue
line_items.filter(VALID).groupby("product_name") \
    .agg(sum("quantity").alias("total_quantity"),
         round(sum(col("quantity") * col("unit_price")), 0).alias("total_revenue")) \
    .orderBy(col("total_revenue").desc()).limit(10) \
    .write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_top_products")

# 4) customer spend (ALL customers, not just top 10)
line_items.filter(VALID).groupby("customer_id", "customer_name", "state") \
    .agg(countDistinct("order_id").alias("total_orders"),
         round(sum(col("quantity") * col("unit_price")), 2).alias("total_spent")) \
    .orderBy(col("total_spent").desc()) \
    .write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_customer_spend")

# 5) sales by state
line_items.filter(VALID).groupby("state") \
    .agg(countDistinct("order_id").alias("total_orders"),
         sum("quantity").alias("total_quantity"),
         round(sum(col("quantity") * col("unit_price")), 0).alias("total_revenue")) \
    .orderBy(col("total_revenue").desc()) \
    .write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_state_sales")

print("5 gold tables written")

In [0]:
# Let me check if the Delta tables actually got created and the data looks right

# First, list every table in the workspace.default schema
all_tables = spark.catalog.listTables("workspace.default")

# Keep only the tables I created (bronze_, silver_, gold_)
created = []
for t in all_tables:
    name = t.name
    if name.startswith("bronze_") or name.startswith("silver_") or name.startswith("gold_"):
        created.append(name)

# Show what I found
print("Delta tables created:", len(created))
for name in sorted(created):
    print("  -", name)

# Check 1: did dedup work? silver should have 2000 rows (not 2015)
print("silver_customers rows:", spark.table("workspace.default.silver_customers").count(), "(2000 = dedup OK)")

# Check 2: do the gold numbers look right? Let me eyeball the monthly revenue
display(spark.table("workspace.default.gold_monthly_revenue"))

# Ecommerce Sales Analytics — PySpark ETL Project

> My end-to-end ETL + analytics pipeline on Databricks: I take raw e-commerce CSVs, clean and validate them, join all four tables, run four phases of business analysis, and save everything as Delta tables.

---

## Dataset

I loaded raw CSVs from a Unity Catalog volume at workspace.default.tutorial_learning.

| File | Rows | Key Columns |
|---|---:|---|
| customers.csv | 2,015 (2,000 distinct) | customer_id, customer_name, city, state, country, signup_date |
| orders.csv | 10,000 | order_id, customer_id, order_date, payment_method, order_status |
| order_items.csv | 21,968 | order_id, product_id, quantity, unit_price |
| products.csv | 100 | product_id, product_name, category, price |

## Key Insights

* Revenue grew about 9x from 2022 ($4.3M) to 2025 ($39.0M)
* Electronics is the top category — $63.6M (~67% of product revenue)
* UPI is the #1 payment method — 3,823 orders, $40.7M revenue
* About $12.4M in potential lost sales from cancelled + returned orders
* 99.4% customer conversion rate
* Gujarat is the top state ($11.04M revenue — I corrected this after deduplicating 15 customers, which had inflated the numbers by ~$164K)

## Medallion Architecture (Phase 5)

I saved all pipeline results as Delta tables in the `workspace.default` schema:

| Layer | Tables | What's in there |
|---|---|---|
| Bronze | bronze_customers, bronze_orders, bronze_order_items, bronze_products | raw data exactly as I got it (duplicates kept for audit) |
| Silver | silver_customers, silver_orders, silver_order_items, silver_products | deduplicated & validated (2,000 customers) |
| Gold | gold_monthly_revenue, gold_yearly_revenue, gold_top_products, gold_customer_spend, gold_state_sales | dashboard-ready business aggregates |

## Tech Stack

* Databricks Serverless compute
* PySpark DataFrame API
* Spark SQL with temp views
* Window functions (rank())
* Unity Catalog Volumes for raw-file storage
* Delta Lake medallion architecture (Bronze / Silver / Gold layers in Unity Catalog)

## How to Run

1. Attach a Serverless compute to this notebook.
2. Run all cells top-to-bottom.
3. You need read access to /Volumes/workspace/default/tutorial_learning/raw/.
4. Phase 5 writes 13 Delta tables to the workspace.default schema (needs CREATE permission).